# Manual cleaning log

This code creates a manual cleaning log (= cleaning audit) by dirctly comparing the original file with the manually cleaned one. 

In [4]:
import json
import csv
from pathlib import Path
from collections import defaultdict, Counter

# ==================================================
# BASE PATHS
# ==================================================
EXTRACTED_DIR = Path("./../output")
CLEANED_DIR   = Path("./../data/datasets/manually_cleaned")
OUTPUT_DIR    = Path("./../output/cleaning_audit")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ==================================================
# AGENCY NAME NORMALIZATION
# ==================================================
AGENCY_MAP = {
    # Australia
    "TGA": "AUSTRALIA",
    "Australia": "AUSTRALIA",
    "AUSTRALIA": "AUSTRALIA",

    # Japan
    "PMDA": "JAPAN",
    "Japan": "JAPAN",
    "JAPAN": "JAPAN",

    # EMA
    "EMA": "EMA",

    # Swissmedic
    "SwissMedic": "SWISSMEDIC",
    "SWISSMEDIC": "SWISSMEDIC",

    # FDA
    "FDA": "FDA",

    # Health Canada
    "HEALTHCANADA": "HEALTHCANADA",
    "HealthCanada": "HEALTHCANADA",
}

# ==================================================
# HELPER FUNCTIONS
# ==================================================
def normalize_display(value):
    """
    Convert values into a comparable string representation.
    """
    if value is None:
        return "MISSING"
    if isinstance(value, list):
        return "; ".join(map(str, value))
    return str(value).strip()


def extract_agency_from_filename(filename: str) -> str | None:
    """
    Extract and normalize the agency identifier from a filename.
    Expected filename format: <prefix>_<agency>_<suffix>.json
    """
    parts = filename.split("_")
    if len(parts) < 2:
        return None
    raw_agency = parts[1]
    return AGENCY_MAP.get(raw_agency)

# ==================================================
# EMA: MERGE ALL EXTRACTION FILES
# ==================================================
def load_all_extracted_for_agency(extracted_dir: Path, agency: str) -> dict:
    """
    Load and merge all extracted JSON files for a given agency (EMA).
    Supports both dict- and list-based JSON structures.
    """
    merged = {}

    for file in extracted_dir.glob(f"*_{agency}_*.json"):
        with open(file, "r", encoding="utf-8") as fh:
            data = json.load(fh)

        if isinstance(data, dict):
            for key, value in data.items():
                if isinstance(value, dict):
                    merged[key] = value

        elif isinstance(data, list):
            for i, record in enumerate(data):
                if not isinstance(record, dict):
                    continue
                key = (
                    record.get("Document_name")
                    or record.get("document_name")
                    or f"{agency}_{file.stem}_{i}"
                )
                merged[key] = record

    return merged

# ==================================================
# CORE AUDIT LOGIC
# ==================================================
def build_audit(original: dict, cleaned: dict) -> dict:
    """
    Compare original and manually cleaned records and
    collect all value changes per field.
    """
    audit = defaultdict(lambda: defaultdict(Counter))
    common_keys = set(original.keys()) & set(cleaned.keys())

    for doc_key in common_keys:
        original_record = original.get(doc_key)
        cleaned_record  = cleaned.get(doc_key)

        if not isinstance(original_record, dict):
            continue
        if not isinstance(cleaned_record, dict):
            continue

        for field, original_value in original_record.items():
            cleaned_value = cleaned_record.get(field)

            original_display = normalize_display(original_value)
            cleaned_display  = normalize_display(cleaned_value)

            if original_display == cleaned_display:
                continue

            audit[field][cleaned_display][original_display] += 1

    return audit

# ==================================================
# CSV OUTPUT
# ==================================================
def write_audit_csv(audit: dict, output_path: Path):
    """
    Write the audit summary to a CSV file.
    """
    with open(output_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow([
            "field",
            "cleaned_value",
            "total_count",
            "original_variants"
        ])

        for field in sorted(audit.keys()):
            for cleaned_value, counter in audit[field].items():
                total = sum(counter.values())
                variants = "; ".join(
                    f"{original} ({count})"
                    for original, count in counter.most_common()
                )

                writer.writerow([
                    field,
                    cleaned_value,
                    total,
                    variants
                ])

# ==================================================
# SPECIAL CASES: FDA & HEALTH CANADA
# ==================================================
EXTRA_AGENCIES = {
    "FDA": {
        "original": Path("./../data/FDA/FDA.json"),
        "cleaned": Path("./../data/datasets/manually_cleaned/FDA_manually_cleaned.json"),
    },
    "HEALTHCANADA": {
        "original": Path("./../data/HealthCanada/HEALTHCANADA.json"),
        "cleaned": Path("./../data/datasets/manually_cleaned/HEALTHCANADA_manually_cleaned.json"),
    },
}

for agency, paths in EXTRA_AGENCIES.items():
    print(f"Processing {agency}")

    if not paths["original"].exists():
        print(f"⚠️  Original file missing for {agency}")
        continue
    if not paths["cleaned"].exists():
        print(f"⚠️  Cleaned file missing for {agency}")
        continue

    with open(paths["original"], "r", encoding="utf-8") as f:
        raw_original = json.load(f)
    original_data = {k: v for k, v in raw_original.items() if isinstance(v, dict)}

    with open(paths["cleaned"], "r", encoding="utf-8") as f:
        raw_cleaned = json.load(f)
    cleaned_data = {k: v for k, v in raw_cleaned.items() if isinstance(v, dict)}

    audit = build_audit(original_data, cleaned_data)

    output_csv = OUTPUT_DIR / f"{agency}_cleaning_audit.csv"
    write_audit_csv(audit, output_csv)

    print(f"   ✅ Saved: {output_csv}")

# ==================================================
# STANDARD AGENCIES FROM EXTRACTED_DIR
# ==================================================
for extracted_file in EXTRACTED_DIR.glob("*.json"):
    agency = extract_agency_from_filename(extracted_file.name)
    if agency is None:
        print(f"⚠️  Agency not recognized: {extracted_file.name}")
        continue

    # FDA and Health Canada already processed above
    if agency in EXTRA_AGENCIES:
        continue

    cleaned_file = CLEANED_DIR / f"{agency}_manually_cleaned.json"
    if not cleaned_file.exists():
        print(f"⚠️  No cleaned file for {agency} – skipped")
        continue

    print(f"Processing {agency}")

    if agency == "EMA":
        original_data = load_all_extracted_for_agency(EXTRACTED_DIR, "EMA")
    else:
        with open(extracted_file, "r", encoding="utf-8") as f:
            raw_original = json.load(f)
        original_data = {k: v for k, v in raw_original.items() if isinstance(v, dict)}

    with open(cleaned_file, "r", encoding="utf-8") as f:
        raw_cleaned = json.load(f)
    cleaned_data = {k: v for k, v in raw_cleaned.items() if isinstance(v, dict)}

    audit = build_audit(original_data, cleaned_data)

    output_csv = OUTPUT_DIR / f"{agency}_cleaning_audit.csv"
    write_audit_csv(audit, output_csv)

    print(f"   ✅ Saved: {output_csv}")

print("\nAll available agencies processed successfully.")


Processing FDA
   ✅ Saved: ../output/cleaning_audit/FDA_cleaning_audit.csv
Processing HEALTHCANADA
   ✅ Saved: ../output/cleaning_audit/HEALTHCANADA_cleaning_audit.csv
Processing SWISSMEDIC
   ✅ Saved: ../output/cleaning_audit/SWISSMEDIC_cleaning_audit.csv
Processing EMA
   ✅ Saved: ../output/cleaning_audit/EMA_cleaning_audit.csv
Processing AUSTRALIA
   ✅ Saved: ../output/cleaning_audit/AUSTRALIA_cleaning_audit.csv
Processing JAPAN
   ✅ Saved: ../output/cleaning_audit/JAPAN_cleaning_audit.csv

All available agencies processed successfully.
